# CE541E08 — Unit 4 · Day 33 — Function Application — apply() and map()

| **CO** | CO4 | **Topics** | apply() · map() · Rational Method row-wise · custom functions |
|---|---|---|---|

---

In [ ]:
student_name="Your Name"; roll="2024XXXXXX"
print(f"CE541E08 | Day 33 | {student_name} | {roll}")

### ▶ Cell 1 — apply() row-wise Rational Method

In [ ]:
import pandas as pd, numpy as np

catchments = pd.DataFrame({
    'Catchment'   : ['A','B','C','D','E'],
    'Area_ha'     : [125, 89, 234, 178, 95],
    'C_runoff'    : [0.65, 0.55, 0.70, 0.60, 0.50],
    'Intensity_mmhr': [52, 52, 52, 52, 52],
})

def rational_Q(row):
    A = row['Area_ha'] * 10000   # ha to m2
    i = row['Intensity_mmhr'] / 1000 / 3600   # mm/hr to m/s
    C = row['C_runoff']
    return round(C * i * A, 4)

catchments['Q_m3s'] = catchments.apply(rational_Q, axis=1)
catchments['Q_Ls']  = catchments['Q_m3s'] * 1000
print(catchments)
print(f"Total peak Q: {catchments['Q_m3s'].sum()*1000:.2f} L/s")
# --- INSTRUCTOR NOTE ---
# axis=1 applies the function to each ROW
# axis=0 would apply to each COLUMN
# apply() is the bridge between Pandas and custom Python functions

### ▶ Cell 2 — map(): replace codes with labels

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Date'    : ['2024-07-01','2024-07-02','2024-07-03','2024-07-04','2024-07-05'],
    'Flow_m3s': [234.5, 678.9, 1234.5, 456.7, 890.2],
    'Soil_code': ['A','B','C','B','A'],
    'QA_code'  : [1, 1, 2, 1, 3],
})

soil_map = {'A':'Sandy loam (low CN)', 'B':'Clay loam (medium CN)', 'C':'Clay (high CN)'}
qa_map   = {1:'GOOD', 2:'SUSPECT', 3:'HIGH'}

df['Soil_type'] = df['Soil_code'].map(soil_map)
df['QA_Flag']   = df['QA_code'].map(qa_map)
print(df[['Date','Flow_m3s','Soil_type','QA_Flag']])
# --- INSTRUCTOR NOTE ---
# .map() replaces each value using a dictionary
# More readable than a chain of df['col'].replace()

### ▶ Cell 3 — apply() for IMD classification

In [ ]:
import pandas as pd, numpy as np

np.random.seed(1)
df = pd.DataFrame({
    'Date'       : pd.date_range('2024-07-01',periods=30,freq='D'),
    'Rainfall_mm': np.round(np.random.exponential(30,30),1),
})

def imd_category(rain):
    if rain == 0:           return 'No rain'
    elif rain <= 15.5:      return 'Light'
    elif rain <= 64.4:      return 'Moderate'
    elif rain <= 115.5:     return 'Heavy'
    elif rain <= 204.4:     return 'Very Heavy'
    else:                   return 'Extremely Heavy'

df['IMD_Cat'] = df['Rainfall_mm'].apply(imd_category)
print(df.head(10))
print()
print("Category counts:")
print(df['IMD_Cat'].value_counts())

### ▶ Cell 4 — apply() with SCS-CN

In [ ]:
import pandas as pd, numpy as np

np.random.seed(3)
df = pd.DataFrame({
    'Storm_ID': range(1,21),
    'P_mm'    : np.round(np.random.uniform(20,150,20),1),
    'CN'      : np.random.choice([65,70,75,80,85],20),
})

def scs_runoff(row):
    P  = row['P_mm']
    CN = row['CN']
    S  = 25400/CN - 254
    Ia = 0.2*S
    if P > Ia:
        return round((P-Ia)**2/(P-Ia+S),2)
    return 0.0

df['Q_mm']      = df.apply(scs_runoff, axis=1)
df['Runoff_pct'] = (df['Q_mm']/df['P_mm']*100).round(1)
print(df.head(10))
print(f"Mean runoff ratio: {df['Runoff_pct'].mean():.1f}%")

---
## Day 33 Assignment
DataFrame of 8 pipe sections: Section, Diameter_mm, Length_m, Slope.
Using apply(): compute velocity (Manning's n=0.013) and head loss (Darcy-Weisbach f=0.018) for each row.
Flag sections where velocity is out of range (0.6–3.0 m/s).

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np, math
df = pd.DataFrame({
    'Section'    : list('ABCDEFGH'),
    'Diameter_mm': [200,250,300,200,250,300,350,400],
    'Length_m'   : [100,150,200,80,120,180,250,300],
    'Slope'      : [0.002,0.003,0.001,0.004,0.002,0.0015,0.001,0.001],
})
def pipe_calc(row):
    D = row['Diameter_mm']/1000; n=0.013; f=0.018; g=9.81
    A = math.pi*(D/2)**2; R = D/4
    V = (1/n)*R**(2/3)*row['Slope']**0.5
    hf = f*(row['Length_m']/D)*(V**2/(2*g))
    return pd.Series({'V_ms':round(V,3),'hf_m':round(hf,3)})
results = df.apply(pipe_calc, axis=1)
df = pd.concat([df, results], axis=1)
df['OK'] = df['V_ms'].between(0.6, 3.0)
print(df)

---
- [ ] Upload: `Unit4_Pandas/CE541E08_U4_Day33.ipynb`

*CE541E08 · Civil Engineering · Christ University*